<a href="https://colab.research.google.com/github/edaska/Stochastic_Processes_-_Optimization_in_Machine_Learning/blob/main/lab9/Stochastics_Lab9_Part1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Radial Basis Function

The scope of this exercise is to predict whether a bank customer will be eligible for a term deposit. The prediction will be based on selected demographic and personal characteristics, such as age, occupation, education level, marital status, and other relevant features.

To perform this task, we will use a neural-network approach based on Radial Basis Functions (RBF). In this model, the centers of the hidden-layer neurons are determined using the k-means clustering algorithm.




Step 1: Loading the required libraries.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import math
from sklearn.metrics import accuracy_score

**Step 2**: Loading of the bank-full.csv dataset and keeping only the columns that will be used as input features, such as the customer’s job, loan status, and other relevant attributes. The dataset is available [here](https://raw.githubusercontent.com/netmode/Stochastic-Processes-and-Optimization-in-Machine-Learning-Lab/refs/heads/main/lab9/bank-full.csv)
. Download and upload to your Colab environment before running the code.

In [19]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
dataset = pd.read_csv('/content/drive/MyDrive/NTUA FINANCIAL ENGINEERING MASTER/2nd Semester/Stochastic Processes & Optimization in Machine Learning/Lab9/bank-full3.csv',sep=None)
print(dataset)

/tmp/ipykernel_3467/538250012.py:1: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support sep=None with delim_whitespace=False; you can avoid this warning by specifying engine='python'.
  dataset = pd.read_csv('/content/drive/MyDrive/NTUA FINANCIAL ENGINEERING MASTER/2nd Semester/Stochastic Processes & Optimization in Machine Learning/Lab9/bank-full3.csv',sep=None)


       age           job   marital  education default  balance housing loan  \
0       58    management   married   tertiary      no     2143     yes   no   
1       44    technician    single  secondary      no       29     yes   no   
2       33  entrepreneur   married  secondary      no        2     yes  yes   
3       47   blue-collar   married    unknown      no     1506     yes   no   
4       33       unknown    single    unknown      no        1      no   no   
...    ...           ...       ...        ...     ...      ...     ...  ...   
45206   51    technician   married   tertiary      no      825      no   no   
45207   71       retired  divorced    primary      no     1729      no   no   
45208   72       retired   married  secondary      no     5715      no   no   
45209   57   blue-collar   married  secondary      no      668      no   no   
45210   37  entrepreneur   married  secondary      no     2971      no   no   

         contact  day month  duration  campaign  pd

In [24]:
dataset.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


In [25]:
col_to_use = ['age','balance','day','duration','campaign','pdays','previous']
data = dataset.drop(col_to_use,axis=1)

In [26]:
data.head()

,job,marital,education,default,housing,loan,contact,month,poutcome,y
0,management,married,tertiary,no,yes,no,unknown,may,unknown,no
1,technician,single,secondary,no,yes,no,unknown,may,unknown,no
2,entrepreneur,married,secondary,no,yes,yes,unknown,may,unknown,no
3,blue-collar,married,unknown,no,yes,no,unknown,may,unknown,no
4,unknown,single,unknown,no,no,no,unknown,may,unknown,no


In [27]:
data = data.apply(LabelEncoder().fit_transform)

In [28]:
data.head()

,job,marital,education,default,housing,loan,contact,month,poutcome,y
0,4,1,2,0,1,0,2,8,3,0
1,9,2,1,0,1,0,2,8,3,0
2,2,1,1,0,1,1,2,8,3,0
3,1,1,3,0,1,0,2,8,3,0
4,11,2,3,0,0,0,2,8,3,0


In [29]:
data['job'].unique()

array([ 4,  9,  2,  1, 11,  5,  0,  7,  6, 10,  3,  8])

In [30]:
dataset['job'].unique()

array(['management', 'technician', 'entrepreneur', 'blue-collar',
       'unknown', 'retired', 'admin.', 'services', 'self-employed',
       'unemployed', 'housemaid', 'student'], dtype=object)

In [31]:
data_rest = dataset[col_to_use]
data_rest.head()

,age,balance,day,duration,campaign,pdays,previous
0,58,2143,5,261,1,-1,0
1,44,29,5,151,1,-1,0
2,33,2,5,76,1,-1,0
3,47,1506,5,92,1,-1,0
4,33,1,5,198,1,-1,0


In [32]:
dataset2 = pd.concat([data_rest,data],axis=1)

In [33]:
dataset2.head()

,age,balance,day,duration,campaign,pdays,previous,job,marital,education,default,housing,loan,contact,month,poutcome,y
0,58,2143,5,261,1,-1,0,4,1,2,0,1,0,2,8,3,0
1,44,29,5,151,1,-1,0,9,2,1,0,1,0,2,8,3,0
2,33,2,5,76,1,-1,0,2,1,1,0,1,1,2,8,3,0
3,47,1506,5,92,1,-1,0,1,1,3,0,1,0,2,8,3,0
4,33,1,5,198,1,-1,0,11,2,3,0,0,0,2,8,3,0


**Step 3**: Spliting of the dataset into training and test sets, and scaling of the features using the StandardScaler() function.

In [34]:
X= dataset2.drop('y',axis=1)
y= dataset2['y']

X_train, X_test, y_train, y_test= train_test_split(X,y, test_size= 0.33, random_state= 4)

In [35]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [36]:
X_train

array([[ 3.39736083, -0.45852779,  0.14087303, ..., -0.71332324,
        -1.50513854,  0.44672656],
       [-0.27605947,  2.43941798,  0.50067953, ...,  0.40010942,
        -1.83733874, -2.58685457],
       [ 0.66584317, -0.13326522, -0.93854645, ..., -0.71332324,
         1.48466323, -0.56446715],
       ...,
       [-0.74701079, -0.22619739,  0.62061503, ..., -0.71332324,
         1.15246303,  0.44672656],
       [ 1.60774581, -0.42275894, -1.53822394, ...,  1.51354209,
         0.15586244,  0.44672656],
       [ 0.28908211, -0.13627382,  1.58009901, ..., -0.71332324,
        -0.17633776,  0.44672656]])

**Step 4**: Determining the centers of the hidden-layer neurons using the k-means algorithm. First, we apply k-means to the training data to find the cluster centers, and then compute the standard deviation of the clusters.

In [37]:
K_cent= 8
km= KMeans(n_clusters= K_cent, max_iter= 100)
km.fit(X_train)
cent= km.cluster_centers_

In [38]:
max=0
for i in range(K_cent):
	for j in range(K_cent):
		d= np.linalg.norm(cent[i]-cent[j])
		if(d> max):
			max= d
d= max

sigma= d/math.sqrt(2*K_cent)

**Step 5**: Construction of the F matrix, where each row corresponds to an input sample elements and each column represents the output of one of the K radial basis functions.

In [39]:
shape= X_train.shape
row= shape[0]
column= K_cent
F= np.empty((row,column), dtype= float)

In [40]:
for i in range(row):
  for j in range(column):
    dist= np.linalg.norm(X_train[i]-cent[j])
    F[i][j]= math.exp(-math.pow(dist,2)/math.pow(2*sigma,2))

**Step 6**: Calculation of the weight matrix W.

In [41]:
FTG= np.dot(F.T,F)
FTG_inv= np.linalg.inv(FTG)
fac= np.dot(FTG_inv,F.T)
W= np.dot(fac,y_train)

**Step 7**: Building of the F matrix using the test dataset.

In [42]:
row= X_test.shape[0]
column= K_cent
F_test= np.empty((row,column), dtype= float)
for i in range(row):
	for j in range(column):
		dist= np.linalg.norm(X_test[i]-cent[j])
		F_test[i][j]= math.exp(-math.pow(dist,2)/math.pow(2*sigma,2))

**Step 8**: Evaluation of the prediction accuracy on the test dataset.

In [43]:
prediction= np.dot(F_test,W)
prediction= 0.5*(np.sign(prediction-0.5)+1)

score= accuracy_score(prediction,y_test)
print(score)

0.8865281501340483


## Questions

**Question 1:** Briefly describe how a Radial Basis Function (RBF) neural network works.

**Question 2:** What methods can be used to determine the weights in an RBF network? Explain the main differences between these methods.

**Question 3:** What is the role of the σ (sigma) parameter in an RBF network? Explain how it affects the model’s performance. What may happen if σ is set to a very small or very large value?

**Question 1: Briefly describe how a Radial Basis Function (RBF) neural network works.**

An RBF neural network is a feed-forward network with an input layer, one hidden layer, and an output layer. The hidden layer transforms the input vector using radial basis functions, usually Gaussian functions based on the Euclidean distance from selected centers:

$φ_j(\textbf{x}) = exp(-||\textbf{x}-μ_j||^2)$

So, each hidden neuron responds strongly when the input is close to its center. The output layer then combines these responses linearly:

$y=F(\textbf{x})=\sum_{j=1}^{K} w_jφ(\textbf{x}-μ_j)$

In practice, RBF networks use hybrid learning: first, the centers are chosen, often with K-Means clustering, and then the output weights are learned using supervised learning.

**Question 2: What methods can be used to determine the weights in an RBF network? Explain the main differences between these methods.**

The weights in an RBF network can mainly be determined in two ways:
1. *Direct supervised solution*

If the RBF network uses one hidden node for each training sample, then the output is written as:

$F(\textbf{x})=\sum_{j=1}^{N} w_jφ(\textbf{x}-\textbf{x}_j)$

The weights $w_j$ are found by solving a linear system of equations so that the network output matches the desired labels:

$F(\textbf{x}_i)=d_i$

This method can fit the training data exactly, but it requires many hidden nodes and can become computationally expensive.

2. *Approximate / hybrid learning method*

In practice, instead of using $N$ hidden nodes, we use fewer hidden nodes $K<N$. First, the centers $μ_j$ are selected using an unsupervised method such as K-Means clustering. Then, the output weights $w_j$ are learned usging supervised learning, usually minimizing the MSE between the desired output and the predicted output.

The *main difference* between direct and hybrid method is caused by the hidden nodes. The direct method uses many hidden nodes and can fit the training data exactly, but it needs more storage and computation. The hybrid method uses fewer hidden nodes, is faster and more practical, but gives an approximate solution instead of an exact interpolation.